# LLM-MedQA: GPU-Accelerated Offline Embedding

This notebook is designed to run on a free GPU cloud environment (like Kaggle T4 x2 or Google Colab T4).
It offloads the heavy embedding workload (Stage 3 of the Benchmark Pipeline) from your local CPU to a cloud GPU.

### Workflow:
1. **Upload** your `_normalized_combined.jsonl` (or any valid MedQA JSONL) to the Kaggle Input directory.
2. **Run** this notebook from top to bottom.
3. **Download** the resulting `chunks_with_embeddings.pkl` from the Kaggle Output directory.
4. **Ingest** locally using the modified `benchmark_pipeline.py --precomputed <file>` flag.

## 1. Install Dependencies

In [ ]:
!pip install -q fastembed-gpu tqdm

## 2. Core Logic (Chunking & Embedding)

In [ ]:
import os
import re
import json
import uuid
import pickle
import hashlib
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass
from tqdm.auto import tqdm
from fastembed import TextEmbedding

# ─── SCHEMAS ─────────────────────────────────────────────────────────

@dataclass
class DocumentRecord:
    doc_id: str
    title: str
    body: str
    source_name: str
    url: str = ""
    updated_at: str = ""
    specialty: str = ""
    audience: str = ""
    trust_tier: int = 3

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "DocumentRecord":
        return cls(
            doc_id=d.get("doc_id", str(uuid.uuid4())),
            title=d.get("title", ""),
            body=d.get("body", ""),
            source_name=d.get("source_name", ""),
            url=d.get("url", ""),
            updated_at=d.get("updated_at", ""),
            specialty=d.get("specialty", ""),
            audience=d.get("audience", ""),
            trust_tier=int(d.get("trust_tier", 3))
        )

@dataclass
class Chunk:
    id: str
    text: str
    metadata: Dict[str, Any]
    embedding: List[float] = None

# ─── CHUNKING LOGIC ──────────────────────────────────────────────────

def generate_stable_id(text: str, metadata: dict) -> str:
    raw = f"{text}::{metadata.get('doc_id','')}::{metadata.get('source_name','')}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

def chunk_by_structure(
    text: str,
    title: str = "",
    source_name: str = "",
    updated_at: str = "",
    audience: str = "",
    chunk_size: int = 900,
    overlap: int = 150,
) -> List[Tuple[str, str]]:
    """
    Splits medical text preserving section headers.
    Replicated from LLM-MedQA app.ingest logic.
    """
    import re
    header_pattern = re.compile(r"^(#{1,4})\s+(.+)$", re.MULTILINE)
    
    blocks = []
    last_idx = 0
    current_heading = title
    
    for m in header_pattern.finditer(text):
        start = m.start()
        if start > last_idx:
            chunk_text = text[last_idx:start].strip()
            if chunk_text:
                blocks.append((current_heading, chunk_text))
        current_heading = m.group(2).strip()
        last_idx = m.end()
        
    if last_idx < len(text):
        chunk_text = text[last_idx:].strip()
        if chunk_text:
            blocks.append((current_heading, chunk_text))
            
    if not blocks:
        blocks = [(title, text.strip())]
        
    final_chunks = []
    for heading, content in blocks:
        words = content.split()
        if len(words) <= chunk_size:
            final_chunks.append((heading, content))
            continue
            
        idx = 0
        while idx < len(words):
            end_idx = min(idx + chunk_size, len(words))
            chunk_content = " ".join(words[idx:end_idx])
            final_chunks.append((heading, chunk_content))
            if end_idx == len(words):
                break
            idx += (chunk_size - overlap)
            
    enriched_chunks = []
    for heading, content in final_chunks:
        header_lines = []
        if title and title not in content: header_lines.append(f"Title: {title}")
        if heading and heading != title: header_lines.append(f"Section: {heading}")
        header_lines.append(f"Source: {source_name}")
        if updated_at: header_lines.append(f"Updated: {updated_at}")
        if audience: header_lines.append(f"Audience: {audience}")
        
        context_header = " | ".join(header_lines)
        enriched_text = f"[{context_header}]\n\n{content}"
        enriched_chunks.append((heading, enriched_text))
        
    return enriched_chunks


## 3. Configure Input/Output and Run

In [ ]:
# === CONFIGURATION ===
# Change this to wherever you uploaded the JSONL file in Kaggle Input
INPUT_FILE = "/kaggle/input/medqa-dataset/_normalized_combined.jsonl"  
OUTPUT_FILE = "/kaggle/working/chunks_with_embeddings.pkl"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
# =====================

if not os.path.exists(INPUT_FILE):
    # Create dummy data for testing if no file provided
    print(f"⚠️ INPUT FILE NOT FOUND: {INPUT_FILE}")
    print("Creating dummy test file...\n")
    INPUT_FILE = "dummy.jsonl"
    with open(INPUT_FILE, "w") as f:
        f.write('{"doc_id": "1", "title": "Test", "source_name": "Fake", "body": "This is a test document."}')

# 1. Initialize GPU Model
print(f"Loading model: {EMBEDDING_MODEL} (GPU Mode)")
embedder = TextEmbedding(model_name=EMBEDDING_MODEL, threads=0) # 0 = auto 

# 2. Read and chunk documents
print("Reading and chunking records...")
chunks: List[Chunk] = []
with open(INPUT_FILE, "r", encoding="-utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
            
        rec = DocumentRecord.from_dict(json.loads(line))
        doc_chunks = chunk_by_structure(
            rec.body, title=rec.title, source_name=rec.source_name, 
            updated_at=rec.updated_at, audience=rec.audience
        )
        
        for heading, chunk_text in doc_chunks:
            meta = {
                "doc_id": rec.doc_id,
                "title": rec.title,
                "source_name": rec.source_name,
                "url": rec.url,
                "updated_at": rec.updated_at,
                "specialty": rec.specialty,
                "audience": rec.audience,
                "trust_tier": rec.trust_tier,
                "section": heading
            }
            chunks.append(Chunk(
                id=generate_stable_id(chunk_text, meta),
                text=chunk_text,
                metadata=meta
            ))

print(f"\nGenerated {len(chunks)} chunks. Starting GPU embedding...")

# 3. Compute Embeddings on GPU
texts = [c.text for c in chunks]
BATCH_SIZE = 256 # Adjust if VRAM allows (T4 can handle 512+ for small models)

embeddings = list(embedder.embed(texts, batch_size=BATCH_SIZE))

# 4. Attach and Save
print("Embedding complete! Packaging into Chunk objects...")
for chunk, emb in zip(chunks, embeddings):
    chunk.embedding = emb.tolist()
    
print(f"Saving {len(chunks)} precomputed chunks to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(chunks, f)

print("\n✅ DONE! You can now download chunks_with_embeddings.pkl from the Output pane.")
